In [ ]:
!pip install gradio langchain-openai langchain-community python-dotenv beautifulsoup4

In [ ]:
import os
import gradio as gr
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document
from dotenv import load_dotenv
# Load environment variables from .env
load_dotenv()

In [ ]:
# Initialize the LLM (OpenRouter / Qwen)
llm = ChatOpenAI(
    model="qwen/qwen3-coder-next",  # or any OpenRouter model
    openai_api_base="https://openrouter.ai/api/v1",
    max_tokens=1000,
    temperature=0,
)

In [ ]:
# Setup Prompts and Chains
parser = StrOutputParser()

prompt_summary = PromptTemplate(
    template="Provide a very short, high-level summary of the following terms and conditions: {text}",
    input_variables=["text"]
)

prompt_offensive = PromptTemplate(
    template="Based on the following terms and conditions, list the top 7 most offensive or risky clauses. Format the output strictly as a bulleted list, where each bullet contains very very brief explanation of the clause: {text} \n ",
    input_variables=["text"]
)

summary_chain = prompt_summary | llm | parser
offensive_chain = prompt_offensive | llm | parser

In [ ]:
# Define analysis and chat helpers

def handle_input_change(method):
    if method == "URL Link":
        return gr.update(visible=True), gr.update(visible=False)
    else:
        return gr.update(visible=False), gr.update(visible=True)

def analyze(method, url, raw_text, progress=gr.Progress(track_tqdm=True)):
    text = ""
    if method == "URL Link":
        if not url:
            yield (
                gr.update(value="", visible=False),
                gr.update(value="**Please enter a URL.**", visible=True),
                gr.update(value="", visible=True),
                gr.update(visible=True),
                gr.update(visible=True),
                gr.update(visible=False),
                [],
                {"text": "", "chat_history": []}
            )
            return
            
        yield (
            gr.update(value="⏳ Scraping webpage...", visible=True),
            gr.update(), gr.update(), gr.update(), gr.update(), gr.update(), gr.update(), gr.update()
        )
        try:
            progress(0.1, desc="Scraping webpage...")
            loader = WebBaseLoader(url)
            documents = loader.load()
            text = documents[0].page_content
        except Exception as e:
            yield (
                gr.update(value="", visible=False),
                gr.update(value=f"**Error loading URL:** {e}", visible=True),
                gr.update(value="", visible=True),
                gr.update(visible=True),
                gr.update(visible=True),
                gr.update(visible=False),
                [],
                {"text": "", "chat_history": []}
            )
            return
    else:
        if not raw_text.strip():
            yield (
                gr.update(value="", visible=False),
                gr.update(value="**Please paste some text.**", visible=True),
                gr.update(value="", visible=True),
                gr.update(visible=True),
                gr.update(visible=True),
                gr.update(visible=False),
                [],
                {"text": "", "chat_history": []}
            )
            return
        text = raw_text

    try:
        yield (
            gr.update(value="⏳ Summarizing legal text...", visible=True),
            gr.update(), gr.update(), gr.update(), gr.update(), gr.update(), gr.update(), gr.update()
        )
        progress(0.4, desc="Summarizing legal text...")
        summary = summary_chain.invoke({"text": text})
        
        yield (
            gr.update(value="⏳ Extracting risky clauses...", visible=True),
            gr.update(), gr.update(), gr.update(), gr.update(), gr.update(), gr.update(), gr.update()
        )
        progress(0.7, desc="Extracting risky clauses...")
        offensive = offensive_chain.invoke({"text": text})
        if not offensive or len(offensive.strip()) <= 5:
            offensive = "No highly offensive terms found."
            
        initial_history = [
            {"role": "system", "content": "You are an AI legal assistant. You must ONLY answer questions directly related to the provided Terms and Conditions document. If the question is irrelevant, refuse to answer politely."},
            {"role": "system", "content": f"Here is the Summary of the document:\n{summary}\n\nHere are the top 7 Offensive Terms:\n{offensive}"}
        ]
        state = {"text": text, "chat_history": initial_history}
        
        progress(1.0, desc="Analysis complete!")
        yield (
            gr.update(value="", visible=False), 
            gr.update(value=summary, visible=True), 
            gr.update(value=offensive, visible=True), 
            gr.update(visible=True), 
            gr.update(visible=True),
            gr.update(visible=False),
            [], 
            state
        )
    except Exception as e:
        yield (
            gr.update(value="", visible=False),
            gr.update(value=f"**Error analyzing:** {e}", visible=True),
            gr.update(value="", visible=True),
            gr.update(visible=True),
            gr.update(visible=True),
            gr.update(visible=False),
            [],
            {"text": "", "chat_history": []}
        )

def chat(user_message, chat_history, state):
    if not state or not state.get("text"):
        return "", chat_history + [{"role": "user", "content": "Please analyze a document first."}, {"role": "assistant", "content": ""}], state

    text = state["text"]
    history = state["chat_history"]

    constrained_message = f"System constraint: You are an AI legal assistant analyzing a specific Terms & Conditions document. You must ONLY answer questions directly related to the document. If the question is irrelevant (like general knowledge, math, geography, etc.), refuse to answer politely. User query: {user_message}"
    history.append({"role": "user", "content": constrained_message})

    try:
        chat_result = llm.invoke(history)
        response = chat_result.content
        history.append({"role": "assistant", "content": response})
        
        new_chat_history = chat_history + [{"role": "user", "content": user_message}, {"role": "assistant", "content": response}]
        state["chat_history"] = history
        
        return "", gr.update(value=new_chat_history, visible=True), state
    except Exception as e:
        return "", chat_history + [{"role": "user", "content": user_message}, {"role": "assistant", "content": f"Error: {e}"}], state


In [ ]:

with gr.Blocks(title="⚖️ Terms & Conditions Analyzer", theme=gr.themes.Default()) as demo:
    gr.Markdown("# ⚖️ Terms & Conditions Analyzer")
    gr.Markdown("Paste the URL of the Terms & Conditions or paste the text directly.")
    
    state = gr.State({"text": "", "chat_history": []})
    
    method_radio = gr.Radio(["URL Link", "Paste Text"], value="URL Link", show_label=False)
    url_box = gr.Textbox(placeholder="Enter the URL of the Terms & Conditions:", show_label=False)
    text_box = gr.Textbox(placeholder="Paste the Terms & Conditions text here:", show_label=False, visible=False, lines=8)
    
    analyze_btn = gr.Button("Analyze Document", variant="primary")
    status_out = gr.Markdown(visible=False)
    
    with gr.Row(visible=False) as results_row:
        with gr.Column():
            gr.Markdown("### 📝 Summary:")
            summary_out = gr.Markdown()
        with gr.Column():
            gr.Markdown("### 🚩 Offensive Terms:")
            offensive_out = gr.Markdown()
            
    with gr.Column(visible=False) as chat_col:
        gr.HTML("<hr>")
        gr.Markdown("### 💬 Chat:")
        
        chatbot = gr.Chatbot(show_label=False, visible=False)
        
        with gr.Row():
            msg_input = gr.Textbox(placeholder="Ask a question about the Terms & Conditions...", show_label=False, scale=9)
            send_btn = gr.Button("Send", scale=1)
        
    method_radio.change(handle_input_change, inputs=[method_radio], outputs=[url_box, text_box])
    
    analyze_btn.click(
        analyze,
        inputs=[method_radio, url_box, text_box],
        outputs=[status_out, summary_out, offensive_out, results_row, chat_col, chatbot, chatbot, state]
    )
    
    send_btn.click(
        chat,
        inputs=[msg_input, chatbot, state],
        outputs=[msg_input, chatbot, state]
    )
    msg_input.submit(
        chat,
        inputs=[msg_input, chatbot, state],
        outputs=[msg_input, chatbot, state]
    )

# Launch the inline dashboard inside the notebook
demo.launch(inline=True)
